In [ ]:
# -*- coding: utf-8 -*-
import os, time, numpy as np, pandas as pd, tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Sequential
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

# ---- GPU setup (run every session once) ----
for g in tf.config.list_physical_devices('GPU'):
    try:
        tf.config.experimental.set_memory_growth(g, True)
    except Exception as e:
        print("Could not set memory growth:", e)
print("Devices:", tf.config.list_physical_devices())

# ---- Config ----
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "code":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results" / "intermediate"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

OUT_CSV = RESULTS_DIR / "nn_table2-3.csv"

TURBINES    = list(range(1, 67))
TESTSET_2018 = list(range(1,47)) + [48,49,50,52] + list(range(54,61)) + list(range(62,67))
FEATURES    = ["wind_speed", "temperature"]
TARGET      = "power"


# ---- ANN hyperparameters ----
UNITS1, UNITS2, UNITS3 = 8, 16, 8
LR, EPOCHS, BATCH_SIZE, SEED = 0.001, 100, 2048, 42
np.random.seed(SEED); tf.random.set_seed(SEED)

def build_model(seed=SEED):
    np.random.seed(seed); tf.random.set_seed(seed)
    m = Sequential([
        layers.Dense(UNITS1, activation='relu'),
        layers.Dense(UNITS2, activation='relu'),
        layers.Dense(UNITS3, activation='relu'),
        layers.Dense(1)
    ])
    m.compile(
        loss=keras.losses.MeanSquaredError(),
        optimizer=keras.optimizers.Adam(learning_rate=LR),
        metrics=['mae','mse']
    )
    return m

def read_csv_safe(path):
    try:
        path = Path(path)
        if path.exists():
            return pd.read_csv(path)
    except Exception:
        pass
    return None

rows = []

for i in TURBINES:
    print(f"\n=== Evaluating Turbine {i} ===")

    # Load 2017 test (for LOO) and 2018 test (if available)
    test17 = read_csv_safe(os.path.join(DATA_PATH, f"Turbine{i}_2017.csv"))
    test18 = read_csv_safe(os.path.join(DATA_PATH, f"Turbine{i}_2018.csv")) if i in TESTSET_2018 else None
    if test17 is None or not set(FEATURES + [TARGET]).issubset(test17.columns):
        print(f"Skipping Turbine {i} (missing 2017 data or columns)")
        continue

    # --- Training data: all other turbines (2017) ---
    train_speed, train_temp, train_power = [], [], []
    for j in TURBINES:
        if j == i: 
            continue
        tr = read_csv_safe(os.path.join(DATA_PATH, f"Turbine{j}_2017.csv"))
        if tr is None or not set(FEATURES + [TARGET]).issubset(tr.columns):
            continue
        tr = tr[FEATURES + [TARGET]].dropna()
        if tr.empty: 
            continue
        train_speed.append(tr["wind_speed"].to_numpy(np.float32))
        train_temp.append(tr["temperature"].to_numpy(np.float32))
        train_power.append(tr["power"].to_numpy(np.float32))

    if len(train_speed) == 0:
        continue

    X_train = np.column_stack([
        np.concatenate(train_speed),
        np.concatenate(train_temp)
    ]).astype(np.float32)
    y_train = np.concatenate(train_power).astype(np.float32).reshape(-1)

    # --- Scale ---
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)

    # --- Train model ---
    t1 = time.time()
    model = build_model()
    model.fit(X_train_s, y_train, epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=1, shuffle=True)
    t2 = time.time()
    train_time = round(t2 - t1, 2)

    # --- Test on 2017 ---
    test17 = test17[FEATURES + [TARGET]].dropna()
    X_te17 = test17[FEATURES].to_numpy(np.float32)
    y_te17 = test17[TARGET].to_numpy(np.float32).reshape(-1)
    X_te17_s = scaler.transform(X_te17)

    p1 = time.time()
    pred17 = model.predict(X_te17_s, batch_size=BATCH_SIZE, verbose=0).reshape(-1)
    pred17 = np.clip(pred17, 0.0, None)
    p2 = time.time()
    pred17_time = round(p2 - p1, 2)
    rmse17 = float(np.sqrt(mean_squared_error(y_te17, pred17)))

    # --- Test on 2018 (if available) ---
    if test18 is not None and set(FEATURES + [TARGET]).issubset(test18.columns):
        test18 = test18[FEATURES + [TARGET]].dropna()
        if not test18.empty:
            X_te18 = test18[FEATURES].to_numpy(np.float32)
            y_te18 = test18[TARGET].to_numpy(np.float32).reshape(-1)
            X_te18_s = scaler.transform(X_te18)

            p3 = time.time()
            pred18 = model.predict(X_te18_s, batch_size=BATCH_SIZE, verbose=0).reshape(-1)
            pred18 = np.clip(pred18, 0.0, None)
            p4 = time.time()
            pred18_time = round(p4 - p3, 2)
            rmse18 = float(np.sqrt(mean_squared_error(y_te18, pred18)))
        else:
            rmse18 = np.nan; pred18_time = np.nan
    else:
        rmse18 = np.nan; pred18_time = np.nan

    # --- Record results ---
    rows.append({
        "Turbine": i,
        "Method": "NN",
        "RMSE_2017": rmse17,
        "RMSE_2018": rmse18,
        "Runtime_train": train_time,
        "Runtime_pred17": pred17_time,
        "Runtime_pred18": pred18_time
    })
    print(pd.DataFrame([rows[-1]]))

# ---- Save ----
results = pd.DataFrame(rows,
    columns=["Turbine","Method","RMSE_2017","RMSE_2018","Runtime_train","Runtime_pred17","Runtime_pred18"])
results.to_csv(OUT_CSV, index=False)
print("\nSaved:", OUT_CSV)
print(results.head())
# ---- Update final results.csv : Table 2 and Table 3, Multi-layer NN row ----
import sys
sys.path.insert(0, str(PROJECT_ROOT / "code"))
from update_final_results import update_final_results
import numpy as np

# Table 2 (2017)
nn_table2_rmse = results["RMSE_2017"].mean() if not results.empty else np.nan
nn_table2_runtime = (
    (results["Runtime_train"] + results["Runtime_pred17"]).mean()
    if not results.empty else np.nan
)

update_final_results(
    method="Multi-layer NN",
    table_id="Table 2",
    rmse=nn_table2_rmse,
    nlpd=np.nan,
    runtime=nn_table2_runtime
)

# Table 3 (2018)
results_2018 = results.dropna(subset=["RMSE_2018"]).copy()

nn_table3_rmse = results_2018["RMSE_2018"].mean() if not results_2018.empty else np.nan
nn_table3_runtime = (
    (results_2018["Runtime_train"] + results_2018["Runtime_pred18"]).mean()
    if not results_2018.empty else np.nan
)

update_final_results(
    method="Multi-layer NN",
    table_id="Table 3",
    rmse=nn_table3_rmse,
    nlpd=np.nan,
    runtime=nn_table3_runtime
)